# Option A Baseline Measurement

Captures the base_rating distribution for every appearance under Option A:
no bonuses, no position modifiers — just `sigmoid(dot_product × impact_scalar)`.

This tells us what the weight-based score is actually producing before any
bonus design decisions are made. Every subsequent tuning decision depends on
understanding this distribution first.

In [9]:
from pathlib import Path
import sys, json, math
import pandas as pd
import numpy as np

project_root = Path("..").resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.services.analytics.match_ratings_service import MatchRatingsService

TEAM_NAME    = "Valencia CF"
MATCHES_PATH = project_root / "tests" / "fixtures" / "testing_data" / "valencia_cf_1" / "matches.json"

with open(project_root / "config" / "performance_weights.json")     as f: weights    = json.load(f)
with open(project_root / "config" / "performance_means_stds.json")  as f: means_stds = json.load(f)
with open(MATCHES_PATH) as f: data = json.load(f)

POSITION_GROUP_MAP = {
    'ST': 'ST', 'LW': 'Winger', 'RW': 'Winger',
    'CM': 'CM', 'CDM': 'CDM',
    'CB': 'CB', 'LB': 'FB', 'RB': 'FB',
}
print(f"Loaded {len(data)} matches")

Loaded 155 matches


In [10]:
# ── Capture service — intercepts base_rating before any modifiers ─────────────
# Overrides _apply_pos_modifiers to return (0, 0), so the pipeline produces
# sigmoid(dot_product × impact_scalar) with no bonus contribution whatsoever.
# The actual captured value is extracted from the sigmoid call directly.

class BaseRatingCaptureService(MatchRatingsService):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._captures = []

    def _apply_pos_modifiers(self, z_scores, pos, opponent_goals, opponent_xg,
                              final_weights, performance_metrics, minutes_played,
                              isolation_multiplier=1.0):
        # Compute the dot product exactly as production does
        dot = self._calculate_dot_product(z_scores=z_scores, weights=final_weights)
        impact = math.sqrt(min(minutes_played, 90.0) / 90.0)
        base_score  = dot * impact
        base_rating = self._apply_sigmoid_transformation(raw_score=base_score)

        self._captures.append({
            'pos': pos,
            'group': POSITION_GROUP_MAP.get(pos, pos),
            'minutes': minutes_played,
            'goals':   performance_metrics.get('goals', 0),
            'assists': performance_metrics.get('assists', 0),
            'dot_product':  dot,
            'impact_scalar': impact,
            'base_score':   base_score,
            'base_rating':  base_rating,
        })
        # Return zeroed modifiers — no bonus, no mastery, nothing
        return 0.0, 0.0

print("BaseRatingCaptureService defined.")

BaseRatingCaptureService defined.


In [11]:
# ── Run on all 155 matches ────────────────────────────────────────────────────

svc = BaseRatingCaptureService(weights, means_stds)

for match in data:
    mo = match['data']
    hl = mo['half_length']
    for perf in match['player_performances']:
        if perf['performance_type'] != 'Outfield':
            continue
        svc.calculate_outfield_rating(perf, mo, hl, TEAM_NAME)

df = pd.DataFrame(svc._captures)
df['scored']   = df['goals']   >= 1
df['assisted'] = df['assists'] >= 1

print(f"Captured {len(df)} outfield appearances across {len(data)} matches")

Captured 2267 outfield appearances across 155 matches


In [12]:
# ── Distribution by position group ───────────────────────────────────────────
# Core question: how much spread does the dot product produce before any bonuses?

groups = ['ST', 'Winger', 'CM', 'CDM', 'CB', 'FB']
sub = df[df['group'].isin(groups)]

stats = (sub.groupby('group')['base_rating']
           .agg(['count', 'mean', 'std',
                 lambda x: x.quantile(0.10),
                 lambda x: x.quantile(0.25),
                 lambda x: x.quantile(0.50),
                 lambda x: x.quantile(0.75),
                 lambda x: x.quantile(0.90)])
           .round(3))
stats.columns = ['n', 'mean', 'std', 'p10', 'p25', 'p50', 'p75', 'p90']
stats = stats.reindex(groups)
print("Base rating distribution by position group (no bonuses):")
print(stats.to_string())

Base rating distribution by position group (no bonuses):
          n   mean    std    p10    p25    p50    p75    p90
group                                                       
ST      264  6.330  0.748  5.627  5.792  6.049  6.664  7.483
Winger  459  6.134  0.544  5.593  5.752  5.951  6.390  6.959
CM      473  6.138  0.542  5.576  5.787  5.987  6.410  6.942
CDM     226  6.217  0.538  5.693  5.819  6.046  6.529  6.984
CB      432  6.186  0.578  5.583  5.821  6.028  6.540  7.005
FB      411  6.177  0.542  5.567  5.786  6.046  6.551  6.924


In [13]:
# ── Scorer vs non-scorer split ────────────────────────────────────────────────
# Shows the natural quality gap between scorers and non-scorers in base_rating.
# This is the baseline the goal bonus will sit on top of.

print("Base rating: scorers vs non-scorers (no bonuses applied)")
print(f"\n{'Group':<10} {'non_scored_mean':>16} {'scored_mean':>13} {'natural_gap':>12}")
print("-" * 55)
for g in ['ST', 'Winger', 'CM', 'CDM', 'CB', 'FB']:
    grp = sub[sub['group'] == g]
    ns  = grp[~grp['scored']]['base_rating'].mean()
    s   = grp[ grp['scored']]['base_rating'].mean()
    if grp['scored'].sum() < 3:
        print(f"{g:<10} {ns:>16.3f} {'(too few)':>13}")
    else:
        print(f"{g:<10} {ns:>16.3f} {s:>13.3f} {s-ns:>12.3f}")

Base rating: scorers vs non-scorers (no bonuses applied)

Group       non_scored_mean   scored_mean  natural_gap
-------------------------------------------------------
ST                    6.008         6.795        0.787
Winger                6.060         6.368        0.308
CM                    6.017         6.589        0.572
CDM                   6.208         6.467        0.259
CB                    6.176         6.630        0.454
FB                    6.177     (too few)


In [14]:
# ── Full percentile table for outfield positions ──────────────────────────────
# Fine-grained view — useful for deciding what bonus values are proportionate.

pctiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]
rows = []
for g in groups:
    grp = sub[sub['group'] == g]['base_rating']
    row = {'group': g}
    for p in pctiles:
        row[f'p{p}'] = round(grp.quantile(p/100), 3)
    row['range_p10_p90'] = round(row['p90'] - row['p10'], 3)
    rows.append(row)

pct_df = pd.DataFrame(rows).set_index('group').reindex(groups)
print("Percentile table (p10→p90 range = effective weight-based spread):")
print(pct_df.to_string())

Percentile table (p10→p90 range = effective weight-based spread):
           p1     p5    p10    p25    p50    p75    p90    p95    p99  range_p10_p90
group                                                                               
ST      5.319  5.535  5.627  5.792  6.049  6.664  7.483  7.767  8.326          1.856
Winger  5.368  5.516  5.593  5.752  5.951  6.390  6.959  7.216  7.860          1.366
CM      5.187  5.472  5.576  5.787  5.987  6.410  6.942  7.251  7.718          1.366
CDM     5.360  5.624  5.693  5.819  6.046  6.529  6.984  7.280  7.854          1.291
CB      5.087  5.382  5.583  5.821  6.028  6.540  7.005  7.226  7.839          1.422
FB      5.269  5.455  5.567  5.786  6.046  6.551  6.924  7.183  7.619          1.357


In [15]:
# ── Minutes distribution check ────────────────────────────────────────────────
# Confirms the impact scalar is compressing cameos correctly under Option A.
# A 20-minute appearance should have a noticeably lower base_rating ceiling.

bins  = [0, 20, 45, 60, 75, 90, 999]
labels= ['≤20', '21-45', '46-60', '61-75', '76-90', '>90']
df['min_bucket'] = pd.cut(df['minutes'], bins=bins, labels=labels)

mins_stats = (df.groupby('min_bucket', observed=True)['base_rating']
                .agg(['count', 'mean', 'std',
                      lambda x: x.quantile(0.25),
                      lambda x: x.quantile(0.75)])
                .round(3))
mins_stats.columns = ['n', 'mean', 'std', 'p25', 'p75']
print("Base rating by minutes played:")
print(mins_stats.to_string())

Base rating by minutes played:
              n   mean    std    p25    p75
min_bucket                                 
≤20         289  5.971  0.218  5.840  6.061
21-45       363  6.023  0.370  5.774  6.174
46-60       149  6.038  0.479  5.698  6.342
61-75       352  6.150  0.538  5.765  6.447
76-90       281  6.258  0.664  5.783  6.720
>90         833  6.341  0.684  5.815  6.810


In [16]:
# ── Summary interpretation ────────────────────────────────────────────────────
# The numbers to read before making any bonus decisions:

outfield = sub[sub['group'].isin(['ST', 'Winger', 'CM', 'CDM'])]
p10  = outfield['base_rating'].quantile(0.10)
p90  = outfield['base_rating'].quantile(0.90)
spread = p90 - p10

print("Key figures for bonus design:")
print(f"  Outfield p10 base_rating : {p10:.3f}")
print(f"  Outfield p90 base_rating : {p90:.3f}")
print(f"  p10→p90 spread           : {spread:.3f}  (weight-based range before any bonuses)")
print()
print(
    "  Sigmoid midpoint         : 6.000  (what z=0 → 5.0 raw → sigmoid gives)"
)
print()
print("Interpretation:")
print(f"  A goal bonus of +X should be read against the {spread:.2f} spread above.")
print( "  If bonuses are larger than ~half that spread, they dominate the weights.")
print( "  If they're smaller than ~0.3, they're barely perceptible in the final rating.")

Key figures for bonus design:
  Outfield p10 base_rating : 5.606
  Outfield p90 base_rating : 7.056
  p10→p90 spread           : 1.450  (weight-based range before any bonuses)

  Sigmoid midpoint         : 6.000  (what z=0 → 5.0 raw → sigmoid gives)

Interpretation:
  A goal bonus of +X should be read against the 1.45 spread above.
  If bonuses are larger than ~half that spread, they dominate the weights.
  If they're smaller than ~0.3, they're barely perceptible in the final rating.
